> Public repository edition. The analysis uses the included **sample** by default. Full report results used the separate full dataset. Code paths and validation were cleaned for publication; original score formulas, pair split, model settings and evaluation definitions are retained. Outputs are cleared. See [reproducibility notes](../docs/reproducibility.md).

# Final model evaluation

#### Comparing all saved model predictions on the test set

In [ ]:
from pathlib import Path
import sys

# Start Jupyter in the repository root or its notebooks/ directory.
project_root = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "project_config.py").is_file()),
    None,
)
if project_root is None:
    raise RuntimeError("Open this notebook from the repository root or notebooks/ folder.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
from project_config import get_paths

paths = get_paths()
print("Data mode:", paths.mode)
print("Input:", paths.input_path)
print("Local processed data:", paths.processed_dir)
print("Local evaluation outputs:", paths.evaluation_dir)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    brier_score_loss,
    log_loss,
    roc_auc_score,
    average_precision_score,
    roc_curve,
    precision_recall_curve
)
from sklearn.calibration import calibration_curve

In [ ]:
data_dir = paths.processed_dir

logistic_path = data_dir / "IdentityMF_Logistic_r_overlap_test_predictions.csv"
isotonic_path = data_dir / "IdentityMF_Isotonic_r_overlap_test_predictions.csv"
expanded_path = data_dir / "IdentityMF_Expanded_logistic_test_predictions.csv"

logistic_df = pd.read_csv(logistic_path)
isotonic_df = pd.read_csv(isotonic_path)
expanded_df = pd.read_csv(expanded_path)

print("Logistic predictions shape:", logistic_df.shape)
print("Isotonic predictions shape:", isotonic_df.shape)
print("Expanded logistic predictions shape:", expanded_df.shape)

logistic_df.head()

Checking that all prediction files match

In [ ]:
id_columns = ["compound_1", "compound_2", "pair_id", "label"]
# Repeated pair IDs alone do not uniquely identify replicate selections.
# Compare every base-data column, preserving row order, before joining predictions.
base_columns = [
    "compound_1", "compound_2", "pair_id", "minS11", "minS22", "within_min",
    "maxS12", "delta_raw", "r_overlap", "delta_prime",
    "delta_prime_reconstructed", "label",
]
pd.testing.assert_frame_equal(logistic_df[base_columns], isotonic_df[base_columns])
pd.testing.assert_frame_equal(logistic_df[base_columns], expanded_df[base_columns])
print("All prediction files match on all base-data columns in the same row order.")


### Create one final evaluation dataframe
Combining model probabilities into one evaluation dataframe

In [ ]:
eval_df = logistic_df[id_columns].copy()

eval_df["logistic_r_probability"] = logistic_df["logistic_r_probability"]
eval_df["isotonic_r_probability"] = isotonic_df["isotonic_r_probability"]
eval_df["expanded_logistic_probability"] = expanded_df["expanded_logistic_probability"]

eval_df.head()

Defining model names and probability columns

In [ ]:
model_probability_columns = {
    "Logistic r_overlap": "logistic_r_probability",
    "Isotonic r_overlap": "isotonic_r_probability",
    "Expanded logistic": "expanded_logistic_probability"
}

y_true = eval_df["label"]
threshold = 0.5

### Build final metrics table
Computing final metrics for all models

In [ ]:
# The legacy key "pr_auc" denotes average precision (AP), as in the report.
def compute_metrics(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    
    specificity = tn / (tn + fp)
    false_positive_rate = fp / (fp + tn)
    
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred),
        "specificity": specificity,
        "false_positive_rate": false_positive_rate,
        "f1_score": f1_score(y_true, y_pred),
        "brier_score": brier_score_loss(y_true, y_prob),
        "log_loss": log_loss(y_true, np.clip(y_prob, 1e-15, 1 - 1e-15)),
        "roc_auc": roc_auc_score(y_true, y_prob),
        "pr_auc": average_precision_score(y_true, y_prob),
        "true_negatives": tn,
        "false_positives": fp,
        "false_negatives": fn,
        "true_positives": tp
    }

In [ ]:
metrics_results = {}

for model_name, prob_col in model_probability_columns.items():
    y_prob = eval_df[prob_col]
    metrics_results[model_name] = compute_metrics(y_true, y_prob, threshold=threshold)

final_metrics_df = pd.DataFrame(metrics_results)

final_metrics_df

Creating a cleaner metrics table

In [ ]:
report_metrics = [
    "brier_score",
    "log_loss",
    "roc_auc",
    "pr_auc",
    "accuracy",
    "precision",
    "recall",
    "specificity",
    "f1_score"
]

report_metrics_df = final_metrics_df.loc[report_metrics].copy()
rounded_report_metrics_df = report_metrics_df.round(6)

rounded_report_metrics_df

### Saving the final metrics table

In [ ]:
output_dir = paths.evaluation_dir
output_dir.mkdir(parents=True, exist_ok=True)

metrics_output_path = output_dir / "final_model_metrics.csv"

rounded_report_metrics_df.to_csv(metrics_output_path)

print("Final metrics table saved to:", metrics_output_path)

### Confusion matrices for all models

In [ ]:
confusion_matrices = {}

for model_name, prob_col in model_probability_columns.items():
    y_prob = eval_df[prob_col]
    y_pred = (y_prob >= threshold).astype(int)
    
    conf_matrix = pd.DataFrame(
        confusion_matrix(y_true, y_pred),
        index=["Actual different (0)", "Actual same (1)"],
        columns=["Predicted different (0)", "Predicted same (1)"]
    )
    
    confusion_matrices[model_name] = conf_matrix
    
    print("\n" + model_name)
    display(conf_matrix)

### Calibration curve comparison

In [ ]:
plt.figure(figsize=(7, 7))

for model_name, prob_col in model_probability_columns.items():
    prob_true, prob_pred = calibration_curve(
        y_true,
        eval_df[prob_col],
        n_bins=10,
        strategy="quantile"
    )
    
    plt.plot(prob_pred, prob_true, marker="o", label=model_name)

plt.plot([0, 1], [0, 1], linestyle="--", label="Perfect calibration")

plt.xlabel("Mean predicted probability")
plt.ylabel("Observed fraction of same-compound rows")
plt.title("Calibration curve comparison")
plt.legend()
plt.grid(alpha=0.3)

calibration_plot_path = output_dir / "calibration_curve_comparison.png"
plt.savefig(calibration_plot_path, dpi=300, bbox_inches="tight")

plt.show()

print("Calibration curve saved to:", calibration_plot_path)

### Plotting ROC curves for all models

In [ ]:
plt.figure(figsize=(7, 6))

for model_name, prob_col in model_probability_columns.items():
    y_prob = eval_df[prob_col]
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    auc_value = roc_auc_score(y_true, y_prob)
    
    plt.plot(fpr, tpr, label=f"{model_name} (AUC = {auc_value:.6f})")

plt.plot([0, 1], [0, 1], linestyle="--", label="Random")

plt.xlabel("False positive rate")
plt.ylabel("True positive rate")
plt.title("ROC curve comparison")
plt.legend()
plt.grid(alpha=0.3)

roc_plot_path = output_dir / "roc_curve_comparison.png"
plt.savefig(roc_plot_path, dpi=300, bbox_inches="tight")

plt.show()

print("ROC curve saved to:", roc_plot_path)

### Precision-recall curve comparison

In [ ]:
plt.figure(figsize=(7, 6))

for model_name, prob_col in model_probability_columns.items():
    y_prob = eval_df[prob_col]
    precision, recall, _ = precision_recall_curve(y_true, y_prob)
    pr_auc = average_precision_score(y_true, y_prob)
    
    plt.plot(recall, precision, label=f"{model_name} (AP = {pr_auc:.6f})")

baseline = y_true.mean()
plt.axhline(baseline, linestyle="--", label=f"Baseline = {baseline:.4f}")

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-recall curve comparison")
plt.legend()
plt.grid(alpha=0.3)

pr_plot_path = output_dir / "precision_recall_curve_comparison.png"
plt.savefig(pr_plot_path, dpi=300, bbox_inches="tight")

plt.show()

print("Precision-recall curve saved to:", pr_plot_path)

### Predicted probability distributions

In [ ]:
for model_name, prob_col in model_probability_columns.items():
    plt.figure(figsize=(8, 5))
    
    for label in sorted(eval_df["label"].unique()):
        subset = eval_df[eval_df["label"] == label]
        plt.hist(
            subset[prob_col],
            bins=50,
            alpha=0.5,
            density=True,
            label=f"label = {label}"
        )
    
    plt.xlabel("Predicted probability")
    plt.ylabel("Density")
    plt.title(f"Predicted probability distribution: {model_name}")
    plt.legend()
    plt.grid(alpha=0.3)
    
    safe_name = model_name.lower().replace(" ", "_")
    plot_path = output_dir / f"probability_distribution_{safe_name}.png"
    plt.savefig(plot_path, dpi=300, bbox_inches="tight")
    
    plt.show()
    
    print("Saved to:", plot_path)

### Final summary table of best models
Identifying the best model for each metric

In [ ]:
lower_is_better = ["brier_score", "log_loss", "false_positive_rate"]

best_model_by_metric = {}

for metric in report_metrics:
    if metric in lower_is_better:
        best_model = report_metrics_df.loc[metric].idxmin()
    else:
        best_model = report_metrics_df.loc[metric].idxmax()
    
    best_model_by_metric[metric] = best_model

best_model_df = pd.DataFrame.from_dict(
    best_model_by_metric,
    orient="index",
    columns=["best_model"]
)

best_model_df

In [ ]:
best_model_output_path = output_dir / "best_model_by_metric.csv"

best_model_df.to_csv(best_model_output_path)

print("Best model table saved to:", best_model_output_path)

## Summarizing the final model choice

- Best *calibrated probability model*: **Isotonic `r_overlap`**
- Best *interpretable model*: **Logistic `r_overlap`**
- **Expanded logistic model** did not improve performance